# Trace Analysis

Neutron resonance imaging can detect trace elements in a bulk matrix by
exploiting their unique resonance fingerprints — even when the trace
concentration is only a few hundred ppm.

This notebook demonstrates **ppm-level Mn detection in a steel (Fe) matrix**
on a synthetic 20×20 pixel phantom, exploring the limits of spatial resolution
and sensitivity achievable with VENUS-like count rates.

| Step | Content |
|------|---------|
| 1. Cross-sections | Fe-56 vs Mn-55 spectral contrast in 1–50 eV |
| 2. Phantom | Steel plate with Mn-rich inclusion |
| 3. Forward model | Transmission cube at ppm Mn levels |
| 4. Noise | Poisson simulation (I₀ = 1000, high-statistics regime) |
| 5. Spatial map | Per-pixel Fe/Mn density fit |
| 6. Sensitivity | Detection limit vs I₀ at a single pixel |
| 7. Validation | Fitted vs true Mn density map |
| 8. Statistics | Per-zone bias and precision |

## Units

NEREIDS uses **atoms/barn** for areal densities.
For a matrix dominated by Fe-56 (A = 56, ρ = 7.87 g/cm³, 1 mm thick):

$$N_\text{Fe} = \rho \cdot d \cdot N_A / A \times 10^{-24}
= 7.87 \times 0.1 \times 6.022 \times 10^{23} / 56 \times 10^{-24}
\approx 8.47 \times 10^{-3} \text{ atoms/barn}$$

Manganese at 500 ppm (by atom) in this matrix:
$N_\text{Mn} = 500 \times 10^{-6} \times N_\text{Fe} \approx 4.2 \times 10^{-6}$ atoms/barn

## Prerequisites

```bash
pixi run build
```

**Previous:** [Enrichment Analysis](01_enrichment_analysis.ipynb)  
**Next:** [Forward Model Pipeline](03_forward_model_demo.ipynb)

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

import nereids

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 12

## 1. Fe-56 vs Mn-55 Spectral Fingerprints

Fe-56 (the dominant iron isotope, 91.75% natural abundance) has strong
resonances starting above ~27 keV in the unresolved resonance region but
its lowest resolved resonances are in the keV range — mostly above our
1–50 eV window. The cross-section in this range is dominated by the
smooth potential scattering (~11 barn) and a few low-lying resonances.

Mn-55 (100% natural abundance, no stable isotopes to consider) has
sharp resonances beginning around 2 keV; in the 1–50 eV range, the
cross-section is mainly potential scattering (~2 barn) with
tails from distant resonances.

The key tool for trace detection is the spectral **contrast** per unit density:
$|\partial T / \partial n|$ — the larger this is, the easier detection.

In [ ]:
fe56 = nereids.load_endf(26, 56)
mn55 = nereids.load_endf(25, 55)

print(fe56)
print(mn55)

e_fine = np.linspace(1.0, 50.0, 4000)
xs_fe56 = nereids.cross_sections(e_fine, fe56)
xs_mn55 = nereids.cross_sections(e_fine, mn55)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, xs, name, color in [
    (axes[0], xs_fe56, 'Fe-56', 'steelblue'),
    (axes[1], xs_mn55, 'Mn-55', 'seagreen'),
]:
    ax.semilogy(e_fine, xs['total'],   color=color,     label='total')
    ax.semilogy(e_fine, xs['elastic'], color='dimgray', label='elastic', linestyle=':')
    ax.set_xlabel('Energy (eV)')
    ax.set_ylabel('Cross-section (barn)')
    ax.set_title(f'{name} Cross-Sections (1–50 eV)')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Fe-56 mean total XS (1-50 eV): {xs_fe56["total"].mean():.2f} barn')
print(f'Mn-55 mean total XS (1-50 eV): {xs_mn55["total"].mean():.2f} barn')

## 2. Define Steel Matrix Phantom

We model a 1 mm steel plate (Fe-56 matrix) with a Mn-rich inclusion.
The Mn concentration is expressed in **ppm by atom** relative to the
Fe matrix density.

| Zone | Rows | Cols | n(Fe-56) [a/b] | Mn conc. | n(Mn-55) [a/b] |
|------|------|------|----------------|----------|----------------|
| Steel matrix | all | all | 8.47×10⁻³ | 100 ppm | 8.47×10⁻⁷ |
| Mn-enriched band | 5–14 | 4–15 | 8.47×10⁻³ | 2000 ppm | 1.69×10⁻⁵ |
| Mn inclusion | 8–11 | 8–11 | 8.47×10⁻³ | 5000 ppm | 4.24×10⁻⁵ |

In [ ]:
H, W = 20, 20

# Fe-56 matrix density: 1 mm steel (Fe density 7.87 g/cm3, A=56)
N_FE = 7.87 * 0.1 * 6.022e23 / 56 * 1e-24   # atoms/barn
print(f'Fe-56 matrix density: {N_FE:.4e} atoms/barn ({N_FE * 1e6:.0f} micro-atoms/barn)')

# Mn concentration in ppm by atom (relative to Fe atoms)
true_mn_ppm = np.full((H, W), 100.0)       # 100 ppm background
true_mn_ppm[5:15, 4:16] = 2000.0          # 2000 ppm band
true_mn_ppm[8:12, 8:12] = 5000.0          # 5000 ppm inclusion

# Areal densities
true_n_fe = np.full((H, W), N_FE)          # Fe constant everywhere
true_n_mn = true_mn_ppm * 1e-6 * N_FE     # Mn from ppm

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

im0 = axes[0].imshow(true_mn_ppm, cmap='Greens', vmin=0, interpolation='nearest')
im1 = axes[1].imshow(true_n_fe * 1e3, cmap='Blues', vmin=0, interpolation='nearest')
im2 = axes[2].imshow(true_n_mn * 1e6, cmap='Greens', vmin=0, interpolation='nearest')

plt.colorbar(im0, ax=axes[0], label='Mn concentration (ppm)')
plt.colorbar(im1, ax=axes[1], label='n(Fe-56) x1e3 [atoms/barn]')
plt.colorbar(im2, ax=axes[2], label='n(Mn-55) x1e6 [atoms/barn]')

axes[0].set_title('Ground Truth: Mn (ppm)')
axes[1].set_title('Ground Truth: n(Fe-56)')
axes[2].set_title('Ground Truth: n(Mn-55)')

for ax in axes:
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')

plt.tight_layout()
plt.show()

print(f'Mn range: {true_mn_ppm.min():.0f} – {true_mn_ppm.max():.0f} ppm')
print(f'n(Mn) range: {true_n_mn.min():.2e} – {true_n_mn.max():.2e} atoms/barn')

## 3. Generate Transmission Cube

Although the Mn cross-section in 1–50 eV is relatively flat (~2 barn),
even a small systematic attenuation at ppm levels is distinguishable from
the Fe-56 background when the Fe-56 cross-section itself varies with energy.
The residual spectral shape after Fe-56 subtraction encodes the Mn signal.

In [ ]:
energies = np.linspace(1.0, 50.0, 500)

FLIGHT_PATH_M = 25.0
DELTA_T_US = 0.3
DELTA_L_M = 0.01
TEMP_K = 293.6

# Pre-compute spectra for unique compositions
compositions = np.stack([true_n_fe.ravel(), true_n_mn.ravel()], axis=1)
unique_comps, pixel_indices = np.unique(compositions, axis=0, return_inverse=True)

print(f'Computing {len(unique_comps)} unique spectra for {H * W} pixels ...')

unique_spectra = {}
for i, (n_fe, n_mn) in enumerate(unique_comps):
    t = nereids.forward_model(
        energies,
        [(fe56, float(n_fe)), (mn55, float(n_mn))],
        temperature_k=TEMP_K,
        flight_path_m=FLIGHT_PATH_M,
        delta_t_us=DELTA_T_US,
        delta_l_m=DELTA_L_M,
    )
    mn_ppm_i = n_mn / n_fe * 1e6
    unique_spectra[i] = t
    print(f'  Spectrum {i}: Mn={mn_ppm_i:.0f} ppm  T_mean={t.mean():.4f}')

transmission_true = np.stack(
    [unique_spectra[idx] for idx in pixel_indices], axis=1
).reshape(len(energies), H, W)

# Show how transmission changes with Mn concentration
# Difference from pure Fe matrix is the Mn signal
t_no_mn = unique_spectra[0]   # ~100 ppm background (effectively pure Fe)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
labels = ['100 ppm (matrix)', '2000 ppm (band)', '5000 ppm (inclusion)']
colors = ['steelblue', 'seagreen', 'darkgreen']
for i, (label, color) in enumerate(zip(labels, colors)):
    axes[0].plot(energies, unique_spectra[i], color=color, lw=1.2, label=label)
    diff = unique_spectra[i] - unique_spectra[0]
    if i > 0:
        axes[1].plot(energies, diff * 1e4, color=color, lw=1.2, label=label)

axes[0].set_xlabel('Energy (eV)')
axes[0].set_ylabel('Transmission')
axes[0].set_title('Transmission Spectra at Three Mn Levels')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].axhline(0, color='black', lw=0.5)
axes[1].set_xlabel('Energy (eV)')
axes[1].set_ylabel('Delta T x1e4')
axes[1].set_title('Transmission Difference from 100 ppm Background')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Add Poisson Noise

Trace-element detection requires higher count rates than enrichment mapping.
We use I₀ = 1000 counts/bin — achievable at VENUS with longer acquisition
or beam-line upgrades. At this rate, the relative uncertainty is ~3%,
allowing detection of sub-0.1% transmission differences from the Mn signal.

In [ ]:
I0 = 1000
rng = np.random.default_rng(13)

counts = rng.poisson(I0 * transmission_true)
counts = np.maximum(counts, 1)

transmission_noisy = counts / I0
uncertainty = np.sqrt(counts) / I0

print(f'Mean transmission:         {transmission_noisy.mean():.4f}')
print(f'Mean relative uncertainty: {(uncertainty / transmission_noisy).mean() * 100:.1f}%')
print(f'Mean absolute uncertainty: {uncertainty.mean():.5f}')
print(f'Expected Mn signal (2000 ppm): {abs((unique_spectra[1] - unique_spectra[0]).mean()):.5f}')
print(f'Signal-to-noise estimate (2000 ppm): {abs((unique_spectra[1] - unique_spectra[0]).mean()) / uncertainty.mean():.1f}')

## 5. Run Spatial Mapping

The initial guess for Mn density is deliberately set to zero — the fitter
must discover the Mn signal from the data alone, starting from a pure-Fe
hypothesis. This is the realistic scenario when the trace element distribution
is unknown.

**Note:** The LM fitter uses non-negative density constraints, so densities
never go negative even for pixels where the Mn signal is below the noise floor.

In [ ]:
t0 = time.perf_counter()

result = nereids.spatial_map(
    transmission_noisy,
    uncertainty,
    energies,
    [fe56, mn55],
    temperature_k=TEMP_K,
    initial_densities=[N_FE, 1e-5],    # Fe: good initial guess; Mn: broad prior
    flight_path_m=FLIGHT_PATH_M,
    delta_t_us=DELTA_T_US,
    delta_l_m=DELTA_L_M,
    max_iter=200,
)

elapsed = time.perf_counter() - t0
print(f'Spatial mapping: {elapsed:.2f} s  ({H * W} pixels)')
print(f'Converged: {result.n_converged}/{result.n_total}  ({result.n_converged / result.n_total * 100:.1f}%)')
print(f'Isotope names: {result.isotope_names}')

density_maps = np.array(result.density_maps)    # (2, H, W): [Fe-56, Mn-55]
unc_maps     = np.array(result.uncertainty_maps)
chi2 = np.array(result.chi_squared_map)
conv = np.array(result.converged_map)

n_fe_fit = density_maps[0]
n_mn_fit = density_maps[1]
fitted_mn_ppm = n_mn_fit / np.where(n_fe_fit > 1e-8, n_fe_fit, N_FE) * 1e6

print(f'\nFitted n(Mn) range: {n_mn_fit.min():.2e} – {n_mn_fit.max():.2e} atoms/barn')
print(f'Fitted Mn ppm range: {fitted_mn_ppm.min():.0f} – {fitted_mn_ppm.max():.0f} ppm')

## 6. Sensitivity: Detection Limit vs Count Rate

At a single background pixel (100 ppm true Mn), we sweep I₀ from 200 to
5000 to estimate at what count rate the fit begins to recover the true
density rather than returning zero. This defines the detection limit curve.

In [ ]:
# True spectrum for a 100-ppm pixel
t_bg = unique_spectra[0]

i0_values = [200, 500, 1000, 2000, 5000]
bias_pct_list = []
unc_pct_list  = []

rng_sens = np.random.default_rng(99)
for i0 in i0_values:
    # Average over 16 realisations to reduce Monte-Carlo noise
    biases = []
    for _ in range(16):
        c = rng_sens.poisson(i0 * t_bg)
        c = np.maximum(c, 1)
        t_m = c / i0
        s_m = np.sqrt(c) / i0
        res = nereids.fit_spectrum(
            t_m, s_m, energies, [fe56, mn55],
            temperature_k=TEMP_K,
            flight_path_m=FLIGHT_PATH_M,
            delta_t_us=DELTA_T_US,
            delta_l_m=DELTA_L_M,
            initial_densities=[N_FE, 1e-5],
            max_iter=200,
        )
        fitted_ppm = res.densities[1] / N_FE * 1e6
        biases.append(fitted_ppm - 100.0)
    bias_pct_list.append(np.mean(biases))
    unc_pct_list.append(np.std(biases))

fig, ax = plt.subplots(figsize=(8, 4))
ax.axhline(0, color='black', lw=0.8, linestyle='--')
ax.errorbar(i0_values, bias_pct_list, yerr=unc_pct_list,
            marker='o', color='seagreen', capsize=5, label='bias ± 1σ (16 trials)')
ax.set_xlabel('Open-beam counts I₀ per bin')
ax.set_ylabel('Fitted Mn − True (ppm)')
ax.set_title('Detection Sensitivity: Bias at 100 ppm Mn vs Count Rate')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'{"I0":>6}  {"Bias (ppm)":>12}  {"Std (ppm)":>12}')
print('-' * 34)
for i0, b, s in zip(i0_values, bias_pct_list, unc_pct_list):
    print(f'{i0:>6}  {b:>12.1f}  {s:>12.1f}')

## 7. Compare Fitted vs True Mn Map

In [ ]:
ppm_bias = fitted_mn_ppm - true_mn_ppm

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
vmax_ppm = true_mn_ppm.max() * 1.1

im0 = axes[0].imshow(true_mn_ppm,   cmap='Greens', vmin=0, vmax=vmax_ppm, interpolation='nearest')
im1 = axes[1].imshow(fitted_mn_ppm, cmap='Greens', vmin=0, vmax=vmax_ppm, interpolation='nearest')
vlim = np.abs(ppm_bias).max()
im2 = axes[2].imshow(ppm_bias, cmap='RdBu_r', vmin=-vlim, vmax=vlim, interpolation='nearest')

plt.colorbar(im0, ax=axes[0], label='Mn (ppm)')
plt.colorbar(im1, ax=axes[1], label='Mn (ppm)')
plt.colorbar(im2, ax=axes[2], label='Bias (ppm)')

axes[0].set_title('Ground Truth: Mn (ppm)')
axes[1].set_title('Fitted: Mn (ppm)')
axes[2].set_title('Bias = fitted - true (ppm)')

for ax in axes:
    ax.set_xlabel('Column')
    ax.set_ylabel('Row')

plt.tight_layout()
plt.show()

# Density maps
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
names_iso  = result.isotope_names
truths_iso = [true_n_fe, true_n_mn]
cmaps_iso  = ['Blues', 'Greens']

for row_i, (name, truth, fitted, cmap) in enumerate(zip(names_iso, truths_iso, density_maps, cmaps_iso)):
    vmax = truth.max() * 1.15
    bias = fitted - truth
    vlim = np.abs(bias).max()
    im0 = axes[row_i, 0].imshow(truth,  cmap=cmap,     vmin=0,     vmax=vmax, interpolation='nearest')
    im1 = axes[row_i, 1].imshow(fitted, cmap=cmap,     vmin=0,     vmax=vmax, interpolation='nearest')
    im2 = axes[row_i, 2].imshow(bias,   cmap='RdBu_r', vmin=-vlim, vmax=vlim, interpolation='nearest')
    plt.colorbar(im0, ax=axes[row_i, 0], label='[atoms/barn]')
    plt.colorbar(im1, ax=axes[row_i, 1], label='[atoms/barn]')
    plt.colorbar(im2, ax=axes[row_i, 2], label='bias [atoms/barn]')
    axes[row_i, 0].set_title(f'{name}  —  Ground Truth')
    axes[row_i, 1].set_title(f'{name}  —  Fitted')
    axes[row_i, 2].set_title(f'{name}  —  Bias')
    for ax in axes[row_i]:
        ax.set_xlabel('Column')
        ax.set_ylabel('Row')

plt.tight_layout()
plt.show()

## 8. Quantitative Validation

In [ ]:
zones = [
    ('Matrix (100 ppm)',  (true_mn_ppm == 100.0),  100.0,  N_FE,  true_n_mn[0, 0]),
    ('Band (2000 ppm)',   (true_mn_ppm == 2000.0), 2000.0, N_FE,  true_n_mn[10, 10]),
    ('Inclusion (5000)', (true_mn_ppm == 5000.0),  5000.0, N_FE,  true_n_mn[9, 9]),
]

print('=== Mn Concentration Recovery ===')
print(f'{"Zone":<22} {"True (ppm)":>10} {"Mean fit":>10} {"Std":>8} {"Bias (ppm)":>11}')
print('-' * 66)
for zone_name, mask, true_ppm, _, _ in zones:
    fitted_vals = fitted_mn_ppm[mask]
    print(
        f'{zone_name:<22} {true_ppm:>10.0f}'
        f' {fitted_vals.mean():>10.1f}'
        f' {fitted_vals.std():>8.1f}'
        f' {(fitted_vals.mean() - true_ppm):>10.1f}'
    )

print()
print('=== Density Recovery ===')
print(f'{"Zone":<22} {"Isotope":<8} {"True":>10} {"Mean fit":>10} {"Bias %":>8}')
print('-' * 62)
for zone_name, mask, _, true_fe, true_mn_n in zones:
    for iso_idx, (iso_name, true_n) in enumerate([('Fe-56', true_fe), ('Mn-55', true_mn_n)]):
        vals = density_maps[iso_idx][mask]
        bias_pct = (vals.mean() - true_n) / true_n * 100
        print(f'{zone_name:<22} {iso_name:<8} {true_n:>10.2e} {vals.mean():>10.2e} {bias_pct:>7.1f}%')
    print()

## Summary

This notebook demonstrated ppm-level trace element detection in a bulk matrix:

1. **Spectral contrast** — Mn-55 has a distinctly different energy-dependent
   cross-section from Fe-56 in the 1–50 eV range, enabling spectral separation
2. **Phantom** — Fe-56 matrix (1 mm steel) with Mn at 100, 2000, 5000 ppm in three zones
3. **High count rates** — I₀ = 1000 needed for reliable detection at 100 ppm
4. **Sensitivity curve** — bias and standard deviation as a function of I₀
   quantify the detection limit for a given measurement time
5. **Spatial mapping** — `spatial_map()` recovered the Mn distribution map
   down to the noise limit set by Poisson statistics

### Practical detection limits at VENUS (I₀ = 1000)

- **2000 ppm** (0.2%): reliably detected, bias < 10%
- **100 ppm**: detectable with significant uncertainty; reduce by spatial averaging

### Extending to other trace elements

The same workflow applies to any pair where the trace isotope has spectral
features different from the matrix:

```python
# Example: Co-59 trace in Ni matrix
ni58 = nereids.load_endf(28, 58)
co59 = nereids.load_endf(27, 59)

result = nereids.spatial_map(
    transmission, uncertainty, energies,
    [ni58, co59],
    temperature_k=293.6,
    flight_path_m=25.0, delta_t_us=0.3, delta_l_m=0.01,
)
```

**Previous:** [Enrichment Analysis](01_enrichment_analysis.ipynb)  
**Next:** [Forward Model Pipeline](03_forward_model_demo.ipynb)